# 06 — Drift monitoring and retraining decision

Compute the PSI between a reference window and a current window. PSI ≥ 0.25 is a signal to consider retraining, not an automatic decision to promote a new model.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

import pandas as pd

from cooling_load.config import load_config
from cooling_load.io import read_frame
from cooling_load.monitoring import drift_report

config = load_config(ROOT / "configs/base.yaml")
features = read_frame(config.path("feature_data"))
features["timestamp"] = pd.to_datetime(features["timestamp"], utc=True)
features = features.sort_values("timestamp")
cutoff = features["timestamp"].quantile(0.7)
reference = features[features.timestamp <= cutoff]
current = features[features.timestamp > cutoff]

In [ ]:
monitored = [
    "cooling_load_kwh_lag_3",
    "cooling_load_kwh_mean_24",
    "outdoor_temperature_c_lag_3",
    "chilled_water_flow_m3h_lag_3",
    "occupancy_proxy_lag_3",
]
report = drift_report(reference, current, monitored)
display(report)
print("Retrain candidates:", report.loc[report.status.eq("retrain"), "feature"].tolist())